In [ ]:
# STEP 0 - run FIRST in a fresh kernel. Stops early (with advice) if the GPU is not healthy.
import os, subprocess, time
import torch
def gpu_health():
    if not torch.cuda.is_available(): raise SystemExit("CUDA is not available in this kernel. Select the 'Python (GPU)' kernel.")
    try:
        a = torch.randn(2048, 2048, device="cuda"); b = (a @ a).sum(); torch.cuda.synchronize()
        x = torch.randn(64, 3, 224, 224, device="cuda"); y = torch.nn.functional.conv2d(x, torch.randn(8, 3, 3, 3, device="cuda")); torch.cuda.synchronize()
        free, total = torch.cuda.mem_get_info(); print(f"GPU OK: {torch.cuda.get_device_name(0)} | free {free/1e9:.1f} of {total/1e9:.1f} GB")
        del a, b, x, y; torch.cuda.empty_cache(); return True
    except Exception as e:
        print("GPU TEST FAILED:", type(e).__name__, str(e)[:200])
        print("1) Kernel > Restart Kernel (a CUDA 'illegal memory access' is sticky until you restart).")
        print("2) Close every other notebook that uses the GPU and check 'nvidia-smi'.")
        print("3) If it fails again in a fresh kernel: reboot, update the NVIDIA driver, reset any GPU overclock/undervolt tool to stock.")
        return False
try: print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total,temperature.gpu", "--format=csv"], capture_output=True, text=True, timeout=20).stdout)
except Exception: pass
assert gpu_health(), "GPU not healthy: follow the advice printed above"
dev = "cuda"

In [ ]:
# STAGE 3 / STEP 1 - setup
import gc, re, json, math, shutil, datetime, hashlib
import numpy as np, pandas as pd, timm, torch
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_curve
from tqdm import tqdm

OUT = r"G:\runs"; S3 = rf"{OUT}\stage3"; RUNS = rf"{S3}\runs"; PRED = rf"{S3}\logits"        # RUNS: checkpoints and histories | PRED: predictions | S3: tables and figures
for p_ in (S3, RUNS, PRED): os.makedirs(p_, exist_ok=True)
E_T, SEEDS, L, BS, N_TRIALS = 4, [0, 1, 2], 6, 64, 6        # E_T: epochs of EVERY tuning and final run | N_TRIALS: configurations per method (identical for all methods)
LABELS6 = ["Cardiomegaly", "Edema", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]
FIND5 = ["Cardiomegaly", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]; IDX5 = [LABELS6.index(l) for l in FIND5]
plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"], "mathtext.fontset": "stix", "font.size": 8,
    "axes.labelsize": 8, "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7, "axes.linewidth": 0.6,
    "lines.linewidth": 1.0, "grid.linewidth": 0.4, "figure.dpi": 150, "pdf.fonttype": 42})
COL1, COL2 = 3.5, 7.16
def save_fig(fig, name):
    fig.savefig(rf"{S3}\{name}.pdf", bbox_inches="tight"); fig.savefig(rf"{S3}\{name}.png", dpi=600, bbox_inches="tight"); plt.show()

MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
def prep(A, idx):
    x = torch.from_numpy(np.ascontiguousarray(A[np.asarray(idx)])).to(dev).float().div(255).unsqueeze(1).repeat(1, 3, 1, 1)
    return (x - MEAN) / STD
def gpu_batch(A, idx, flip=False):                                    # identical augmentation to all your earlier runs: random horizontal flip only
    x = torch.from_numpy(np.ascontiguousarray(A[np.asarray(idx)])).to(dev).float().div(255).unsqueeze(1)
    if flip:
        f = torch.rand(len(idx), device=dev) < 0.5; x = torch.where(f.view(-1, 1, 1, 1), x.flip(-1), x)
    return (x.repeat(1, 3, 1, 1) - MEAN) / STD
@torch.inference_mode()
def predict_logits_idx(m, A, idx, bs=128):
    m.eval(); out = []
    for i in range(0, len(idx), bs):
        with torch.autocast(dev, enabled=(dev == "cuda")): o = m(prep(A, idx[i:i + bs]))
        out.append(o.float().cpu())
    torch.cuda.synchronize(); return torch.cat(out).numpy()
sig = lambda z: 1 / (1 + np.exp(-z))
def fauc(y, s):
    y = np.asarray(y).astype(bool); r = rankdata(s); n1 = y.sum(); n0 = len(y) - n1
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
print("device:", dev, "| tuning results:", S3, "| epochs per run:", E_T, "| trials per method:", N_TRIALS)

In [ ]:
# STAGE 3 / STEP 2 - data. Training sources: A = NIH and B = CheXpert (train + validation). VinBigData (C) is NEVER used for training or selection: it is the unseen test site.
nih = pd.read_csv(r"G:\nih_split.csv"); off = nih.official.values == "test"; te_i = np.where(off)[0]
tr_i = np.where(nih.split_rev.values == "train")[0]; va_i = np.where(nih.split_rev.values == "val")[0]
NIH_A = np.memmap(r"G:\nih_224.u8", dtype=np.uint8, mode="r", shape=(len(nih), 224, 224)); Yn = nih[LABELS6].values.astype(int)
cx = pd.read_csv(r"G:\chexpert_final_clean.csv"); cxt = cx[cx.split == "test"].reset_index(drop=True); cxm = cx[cx.split.isin(["train", "val"])].reset_index(drop=True)
cx_tr = np.where(cxm.split.values == "train")[0]; cx_va = np.where(cxm.split.values == "val")[0]
CX_A = np.memmap(r"G:\cx_trainval_224.u8", dtype=np.uint8, mode="r", shape=(len(cxm), 224, 224)); CX_T = np.load(r"G:\cx_test_224.npy")
Ycm = cxm[LABELS6].values.astype(int); Yc = cxt[LABELS6].values.astype(int)
vb = pd.read_csv(rf"{OUT}\vinbig\vinbig_image_labels.csv"); NVB = len(vb)
VB_A = np.memmap(rf"{OUT}\vinbig\vb_train_224.u8", dtype=np.uint8, mode="r", shape=(NVB, 224, 224))
Yvb = np.zeros((NVB, L), int)
for f in FIND5: Yvb[:, LABELS6.index(f)] = vb[f].values
SRC = {"A": dict(arr=NIH_A, Y=Yn, tr=tr_i, va=va_i), "B": dict(arr=CX_A, Y=Ycm, tr=cx_tr, va=cx_va)}
TG = {"A": dict(name="NIH", A=NIH_A, idx=te_i, Y=Yn[te_i], grp=pd.factorize(nih.patient_id.values[off])[0]),
      "B": dict(name="CheXpert", A=CX_T, idx=np.arange(len(CX_T)), Y=Yc, grp=pd.factorize(cxt.patient_id.values)[0]),
      "C": dict(name="VinBigData", A=VB_A, idx=np.arange(NVB), Y=Yvb, grp=None)}

# groups for GroupDRO: by hospital, or by hospital x view (majority view of that hospital versus all other views; this does not depend on how the AP/PA codes are named)
vA = nih["view"].astype(str).values; vB = cxm["AP/PA"].astype(str).values
majA = pd.Series(vA[tr_i]).value_counts().index[0]; majB = pd.Series(vB[cx_tr]).value_counts().index[0]
GR = {"hospital": {"A": np.zeros(len(nih), int), "B": np.ones(len(cxm), int), "n": 2},
      "hospital_view": {"A": (vA != majA).astype(int), "B": 2 + (vB != majB).astype(int), "n": 4}}
def entropy_baseline(Y):                                       # loss of a model that only knows the prevalence of each finding
    p = np.clip(Y.mean(0), 1e-6, 1 - 1e-6); return float(np.mean(-(p * np.log(p) + (1 - p) * np.log(1 - p))))
HBASE, rows = {}, []
for gname, G in GR.items():
    HBASE[gname] = []
    for k in range(G["n"]):
        Yk = np.concatenate([Yn[tr_i][G["A"][tr_i] == k], Ycm[cx_tr][G["B"][cx_tr] == k]]); HBASE[gname].append(entropy_baseline(Yk)); rows.append(dict(grouping=gname, group=k, images=len(Yk), prior_only_loss=HBASE[gname][-1]))
display(pd.DataFrame(rows).round(4)); print(f"majority views used for hospital_view: NIH = {majA!r}, CheXpert = {majB!r}")
print(f"NIH train/val {len(tr_i)}/{len(va_i)} | CheXpert train/val {len(cx_tr)}/{len(cx_va)} | test sites: NIH {len(te_i)}, CheXpert {len(CX_T)}, VinBigData {NVB}")

In [ ]:
# STAGE 3 / STEP 3 - the tuning plan (locked) and the trial table. Every method gets the SAME number of trials, epochs, data and selection rule.
PLAN = dict(
    version=1, question="With an equal tuning budget, does any of DANN, CORAL or GroupDRO (fixed) beat tuned ERM on the unseen site (VinBigData)?",
    protocol=dict(architecture="DenseNet121, ImageNet initialisation", sources="NIH + CheXpert", epochs_per_run=E_T, trials_per_method=N_TRIALS, final_seeds=SEEDS, batch_size=BS,
                  search="random search; trial 0 of every method is its default configuration; the other trials draw log-uniform values"),
    search_space=dict(common="learning rate log-uniform [3e-5, 3e-4]; weight decay log-uniform [1e-5, 1e-3]; sampler in {natural, balanced}",
                      DANN="adversarial weight log-uniform [0.01, 1]", CORAL="weight log-uniform [0.01, 10] on a scale-free CORAL loss",
                      GroupDRO="step size log-uniform [0.001, 0.1]; groups in {hospital, hospital x view}; group hardness = loss divided by the prior-only loss of that group"),
    selection="highest mean validation AUROC (NIH validation and CheXpert validation, best epoch); VinBigData is never used for selection",
    oracle="selection by VinBigData AUROC is reported separately, labelled as an upper bound that leaks the target",
    primary_endpoint="mean AUROC over Cardiomegaly, Consolidation, Atelectasis, Pneumothorax, Pleural Effusion on VinBigData, averaged over 3 seeds of the selected configuration",
    primary_contrasts="each tuned DG method minus tuned ERM, paired bootstrap over images; Holm over the 3 contrasts", delta=0.01, alpha=0.05, B=2000, seed=0,
    outcomes="BETTER: difference >= delta and Holm p < alpha | WORSE: difference < 0 and Holm p < alpha | otherwise NOT DISTINGUISHABLE (the minimal detectable difference is printed next to every null)")
PF = rf"{S3}\analysis_plan_stage3.json"
phash = lambda p: hashlib.sha256(json.dumps(p, sort_keys=True).encode()).hexdigest()[:16]
if os.path.exists(PF):
    saved = json.load(open(PF)); PLAN = saved["plan"]; assert phash(PLAN) == saved["hash"], "the locked plan was edited after it was written"
    print(f"plan LOADED (locked {saved['created']}, hash {saved['hash']})")
else:
    json.dump(dict(created=str(datetime.datetime.now()), hash=phash(PLAN), plan=PLAN), open(PF, "w"), indent=2); print("plan LOCKED now, hash", phash(PLAN))
NBOOT, DELTA, ALPHA = PLAN["B"], PLAN["delta"], PLAN["alpha"]
assert PLAN["protocol"]["epochs_per_run"] == E_T and PLAN["protocol"]["trials_per_method"] == N_TRIALS, "E_T / N_TRIALS in Step 1 differ from the locked plan"

METHODS = ["ERM", "DANN", "CORAL", "GroupDRO"]; RUN_DIAGNOSTIC = True       # one extra run: the ORIGINAL GroupDRO (raw loss) with default settings, to show what the fix changes
DEFAULT = {"ERM": dict(sampler="natural"), "DANN": dict(sampler="balanced", lam=0.1), "CORAL": dict(sampler="balanced", lam=1.0), "GroupDRO": dict(sampler="balanced", eta=0.01, grouping="hospital")}
loguni = lambda rng, lo, hi: float(np.exp(rng.uniform(np.log(lo), np.log(hi))))
TPF = rf"{S3}\trials_plan.csv"
if os.path.exists(TPF): TR = pd.read_csv(TPF); print("trial table LOADED from", TPF)
else:
    rows = []
    for mi, m in enumerate(METHODS):
        rng = np.random.default_rng(1000 + mi)
        for t in range(N_TRIALS):
            tr = dict(trial_id=f"{m.lower()}_t{t}", method=m, trial=t, lr=1e-4, wd=1e-4, sampler=DEFAULT[m]["sampler"], lam=np.nan, eta=np.nan, grouping="hospital", norm="relative" if m == "GroupDRO" else "", diagnostic=False)
            if m == "DANN": tr["lam"] = DEFAULT[m]["lam"]
            if m == "CORAL": tr["lam"] = DEFAULT[m]["lam"]
            if m == "GroupDRO": tr["eta"] = DEFAULT[m]["eta"]
            if t > 0:
                tr["lr"], tr["wd"], tr["sampler"] = loguni(rng, 3e-5, 3e-4), loguni(rng, 1e-5, 1e-3), str(rng.choice(["natural", "balanced"]))
                if m == "DANN": tr["lam"] = loguni(rng, 0.01, 1.0)
                if m == "CORAL": tr["lam"] = loguni(rng, 0.01, 10.0)
                if m == "GroupDRO": tr["eta"], tr["grouping"] = loguni(rng, 0.001, 0.1), str(rng.choice(["hospital", "hospital_view"]))
            rows.append(tr)
    rows.append(dict(trial_id="groupdro_raw_default", method="GroupDRO", trial=0, lr=1e-4, wd=1e-4, sampler="balanced", lam=np.nan, eta=0.01, grouping="hospital", norm="raw", diagnostic=True))
    TR = pd.DataFrame(rows); TR.to_csv(TPF, index=False); print("trial table CREATED:", TPF)
display(TR.round(5))

In [ ]:
# STAGE 3 / STEP 4 - training framework: the same recipe as all your earlier runs (AdamW, cosine schedule, mixed precision, horizontal flips), now with configurable hyperparameters and the FIXED GroupDRO and CORAL
ckp = lambda tid, s: rf"{RUNS}\{tid}_seed{s}.pt"
hist_path = lambda tid, s: ckp(tid, s)[:-3] + "_history.csv"
def finished(tid, s): return os.path.exists(ckp(tid, s)) and os.path.exists(hist_path(tid, s)) and len(pd.read_csv(hist_path(tid, s))) >= E_T

def sample_batches(mode, seed, ep):
    """yields {source: sorted image indices}. natural: random pooled batches (the mix follows dataset sizes). balanced: the same number of images from each hospital in every batch"""
    rng = np.random.default_rng(seed * 100 + ep); srcs = ["A", "B"]; sizes = [len(SRC[k]["tr"]) for k in srcs]; spe = math.ceil(sum(sizes) / BS)
    if mode == "natural":
        dom = np.concatenate([np.full(n, i) for i, n in enumerate(sizes)]); pos = np.concatenate([SRC[k]["tr"] for k in srcs]); perm = rng.permutation(len(pos))
        for k in range(spe):
            s = perm[k * BS:(k + 1) * BS]
            yield {srcs[i]: np.sort(pos[s][dom[s] == i]) for i in range(2) if (dom[s] == i).any()}
    else:
        for k in range(spe): yield {s_: np.sort(rng.choice(SRC[s_]["tr"], BS // 2, replace=False)) for s_ in srcs}
def make_batch(batch, GRP):
    xs, ys, ds, gs = [], [], [], []
    for i, k in enumerate(["A", "B"]):
        if k not in batch: continue
        idx = batch[k]; xs.append(gpu_batch(SRC[k]["arr"], idx, flip=True)); ys.append(SRC[k]["Y"][idx]); ds.append(np.full(len(idx), i)); gs.append(GRP[k][idx])
    t = lambda a, dt: torch.from_numpy(np.concatenate(a)).to(dev).to(dt)
    return torch.cat(xs), t(ys, torch.float32), t(ds, torch.long), t(gs, torch.long)
def val_scores(m):                                                    # validation AUROC of each SOURCE hospital (never the test sites)
    out = {}
    for k in ("A", "B"):
        s_ = SRC[k]; P = predict_logits_idx(m, s_["arr"], s_["va"]); Yv = s_["Y"][s_["va"]]
        out[k] = float(np.mean([fauc(Yv[:, j], P[:, j]) for j in range(L) if 5 <= Yv[:, j].sum() <= len(Yv) - 5]))
    return out

class GRL(torch.autograd.Function):                                   # gradient reversal for DANN
    @staticmethod
    def forward(ctx, x, lam): ctx.lam = lam; return x.view_as(x)
    @staticmethod
    def backward(ctx, g): return -ctx.lam * g, None
def coral_norm(fa, fb):                                               # scale-free CORAL: squared covariance gap divided by the (detached) squared covariance norms, so the weight has a stable meaning
    ca, cb = torch.cov(fa.T), torch.cov(fb.T); return ((ca - cb) ** 2).sum() / ((ca.detach() ** 2).sum() + (cb.detach() ** 2).sum() + 1e-8)
class GroupDRO:
    """GroupDRO with a hardness measure. 'raw' = the original (group loss); 'relative' = group loss divided by that group's prior-only loss, so groups with a higher label prevalence
    are not automatically called 'harder'. The weights q are updated with the hardness; the training loss is always the q-weighted raw loss."""
    def __init__(self, n, eta, H, norm): self.q = torch.ones(n, device=dev) / n; self.eta = eta; self.H = torch.tensor(H, device=dev, dtype=torch.float32); self.norm = norm
    def __call__(self, ls, g):
        pres, gl = torch.zeros_like(self.q), torch.zeros_like(self.q)
        for k in range(len(self.q)):
            mk = g == k
            if mk.any(): gl[k] = ls[mk].mean(); pres[k] = 1.0
        hard = gl.detach() / self.H if self.norm == "relative" else gl.detach()
        self.q = self.q * torch.exp(self.eta * hard * pres); self.q = self.q / self.q.sum()
        w = self.q * pres; return (w * gl).sum() / w.sum()

def train_trial(tr, seed):
    tid, meth = tr["trial_id"], tr["method"]; path = ckp(tid, seed); last, hist_f = path[:-3] + "_last.pt", hist_path(tid, seed)
    if finished(tid, seed): print("skip (finished):", tid, "seed", seed); return
    torch.manual_seed(seed); np.random.seed(seed); GRP = GR[tr["grouping"]]
    m = timm.create_model("densenet121", pretrained=True, num_classes=L).to(dev)
    disc = nn.Sequential(nn.Linear(m.num_features, 256), nn.ReLU(), nn.Linear(256, 2)).to(dev) if meth == "DANN" else None
    dro = GroupDRO(GRP["n"], float(tr["eta"]), HBASE[tr["grouping"]], tr["norm"]) if meth == "GroupDRO" else None
    opt = torch.optim.AdamW(list(m.parameters()) + (list(disc.parameters()) if disc else []), lr=float(tr["lr"]), weight_decay=float(tr["wd"]))
    spe = math.ceil(sum(len(SRC[k]["tr"]) for k in ("A", "B")) / BS); sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=E_T * spe)
    scaler = torch.amp.GradScaler("cuda"); hist, best, start = [], -1.0, 0
    if os.path.exists(last):
        ck = torch.load(last, map_location=dev, weights_only=False)
        m.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sch.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"]); hist, best, start = ck["hist"], ck["best"], ck["ep"] + 1
        if disc: disc.load_state_dict(ck["disc"])
        if dro: dro.q = ck["q"].to(dev)
        print("resumed, next epoch", start + 1)
    for ep in range(start, E_T):
        m.train(); t0 = time.time(); tl = torch.zeros((), device=dev)
        for k, batch in enumerate(tqdm(sample_batches(tr["sampler"], seed, ep), total=spe, leave=False, desc=f"{tid} seed {seed} epoch {ep + 1}/{E_T}")):
            x, y, dm, g = make_batch(batch, GRP)
            with torch.autocast("cuda"):
                if meth in ("DANN", "CORAL"): f = m.forward_head(m.forward_features(x), pre_logits=True); z = m.get_classifier()(f)
                else: z = m(x)
            ls = F.binary_cross_entropy_with_logits(z.float(), y, reduction="none").mean(1)
            loss = dro(ls, g) if dro else ls.mean()
            if meth == "CORAL": loss = loss + float(tr["lam"]) * coral_norm(f.float()[dm == 0], f.float()[dm == 1])
            if meth == "DANN":
                p_ = (ep * spe + k) / (E_T * spe); lam_t = 2 / (1 + math.exp(-10 * p_)) - 1
                loss = loss + float(tr["lam"]) * F.cross_entropy(disc(GRL.apply(f.float(), lam_t)), dm)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step(); tl += loss.detach()
        sc = val_scores(m); vm, vw = float(np.mean(list(sc.values()))), float(min(sc.values()))
        hist.append(dict(epoch=ep + 1, train_loss=float(tl) / spe, val_A=sc["A"], val_B=sc["B"], val_mean=vm, val_worst=vw, minutes=(time.time() - t0) / 60, q=" ".join(f"{v:.4f}" for v in dro.q.cpu().numpy()) if dro else ""))
        pd.DataFrame(hist).to_csv(hist_f, index=False)
        print(f"{tid} seed {seed} epoch {ep + 1} | loss {hist[-1]['train_loss']:.4f} | val AUROC NIH {sc['A']:.4f} CheXpert {sc['B']:.4f} mean {vm:.4f} | {hist[-1]['minutes']:.1f} min" + (f" | q {hist[-1]['q']}" if dro else ""))
        if vm > best: best = vm; torch.save(m.state_dict(), path)
        torch.save(dict(model=m.state_dict(), opt=opt.state_dict(), sched=sch.state_dict(), scaler=scaler.state_dict(), hist=hist, best=best, ep=ep, disc=disc.state_dict() if disc else None, q=dro.q.cpu() if dro else None), last)
    if os.path.exists(last): os.remove(last)
    del m, opt, sch, scaler, disc, dro; gc.collect(); torch.cuda.empty_cache()

def run_trials(items):                                                # items: list of (trial record, seed)
    for tr, s in items:
        try: train_trial(tr, s)
        except Exception as e:
            print(f"\nFAILED on {tr['trial_id']} seed {s}: {type(e).__name__}: {str(e)[:160]}")
            if "illegal memory" in str(e) or "CUDA error" in str(e): print("CUDA context is broken: Kernel > Restart, rerun Steps 0 to 4, then rerun this step (finished runs are skipped, unfinished ones resume).")
            raise
print("framework ready")

In [ ]:
# STAGE 3 / STEP 5 - run the tuning trials (seed 0). Resumable. Order: all default configurations first, then trial 1 of every method, and so on, so partial results are already comparable.
# Cost per run of 4 epochs, from your A+B log (about 7.4 min per epoch for 230k images): about 30 minutes. 24 trials + 1 diagnostic = about 12 hours.
# smoke test first (a few minutes): set E_T = 1 in Step 1, delete the locked plan file, run the next line, then undo both and delete the erm_t0_seed0 files in the runs folder
# run_trials([(TR.iloc[0].to_dict(), 0)])
recs = TR.to_dict("records"); tune = [(r, 0) for r in recs if (not r["diagnostic"]) or RUN_DIAGNOSTIC]
tune.sort(key=lambda x: (x[0]["trial"], x[0]["diagnostic"], METHODS.index(x[0]["method"])))
status = pd.DataFrame([dict(trial=r["trial_id"], method=r["method"], finished=finished(r["trial_id"], s)) for r, s in tune]); display(status)
todo = [(r, s) for r, s in tune if not finished(r["trial_id"], s)]; print(f"{len(todo)} of {len(tune)} tuning runs still to do")
if todo:
    assert gpu_health(), "GPU not healthy: restart the kernel and rerun Steps 0 to 4 first"
    run_trials(todo)
print("tuning runs finished")

In [ ]:
# STAGE 3 / STEP 6 - model selection with SOURCE data only, then the extra seeds for the selected configurations.
rows = []
for r in TR.to_dict("records"):
    if not finished(r["trial_id"], 0): continue
    h = pd.read_csv(hist_path(r["trial_id"], 0)); i = int(h.val_mean.idxmax())
    rows.append(dict(r, best_epoch=int(h.epoch[i]), val_A=h.val_A[i], val_B=h.val_B[i], val_mean=h.val_mean[i], val_worst=h.val_worst[i], last_epoch_val_mean=h.val_mean.iloc[-1]))
TRS = pd.DataFrame(rows); assert len(TRS), "no finished tuning run yet: run Step 5 first"
core = TRS[~TRS.diagnostic]; missing = [(m, N_TRIALS - int((core.method == m).sum())) for m in METHODS if int((core.method == m).sum()) < N_TRIALS]
assert not missing, f"tuning is not finished for {missing}: selection must wait until every method has all its {N_TRIALS} trials (otherwise the budgets are not equal)"
SEL = {m: core[core.method == m].sort_values("val_mean", ascending=False).iloc[0].to_dict() for m in METHODS}                 # primary rule: best mean validation AUROC on the source hospitals
SEL_W = {m: core[core.method == m].sort_values("val_worst", ascending=False).iloc[0]["trial_id"] for m in METHODS}            # secondary rule: best worst-hospital validation AUROC (reported only)
sel_tab = pd.DataFrame([dict(method=m, selected=SEL[m]["trial_id"], lr=SEL[m]["lr"], wd=SEL[m]["wd"], sampler=SEL[m]["sampler"], lam=SEL[m]["lam"], eta=SEL[m]["eta"], grouping=SEL[m]["grouping"],
                             best_epoch=SEL[m]["best_epoch"], val_mean=SEL[m]["val_mean"], val_worst=SEL[m]["val_worst"], same_choice_by_worst_hospital_rule=(SEL_W[m] == SEL[m]["trial_id"])) for m in METHODS])
display(sel_tab.round(5)); sel_tab.to_csv(rf"{S3}\selection_source_validation.csv", index=False); TRS.to_csv(rf"{S3}\trials_source_validation.csv", index=False)
print("source-validation spread per method (best minus worst trial):", {m: round(float(core[core.method == m].val_mean.max() - core[core.method == m].val_mean.min()), 4) for m in METHODS})
extra = [(TR[TR.trial_id == SEL[m]["trial_id"]].iloc[0].to_dict(), s) for m in METHODS for s in SEEDS if s != 0]; extra = [(r, s) for r, s in extra if not finished(r["trial_id"], s)]
print(f"{len(extra)} extra seed runs for the selected configurations (about 30 minutes each)")
if extra:
    assert gpu_health(), "GPU not healthy: restart the kernel and rerun Steps 0 to 4 first"
    run_trials(extra)
print("selected configurations have all seeds")

In [ ]:
 # STAGE 3 / STEP 7 - predictions of every finished run on the three test sites (A = NIH official test, B = CheXpert test, C = all 15,000 VinBigData images)
assert gpu_health(), "GPU not healthy: restart the kernel and rerun Steps 0 to 4 first"
lpath = lambda tid, s: rf"{PRED}\logits_{tid}_seed{s}.npz"
def load_model(tid, s):
    m = timm.create_model("densenet121", pretrained=False, num_classes=L).to(dev)
    m.load_state_dict(torch.load(ckp(tid, s), map_location=dev, weights_only=True)); return m.eval()
for r in TR.to_dict("records"):
    for s in SEEDS:
        if os.path.exists(lpath(r["trial_id"], s)) or not finished(r["trial_id"], s): continue
        try:
            m = load_model(r["trial_id"], s); z = {T: predict_logits_idx(m, g["A"], g["idx"]) for T, g in TG.items()}
            assert all(np.isfinite(v).all() for v in z.values()), "non-finite logits"; np.savez(lpath(r["trial_id"], s), **z); del m; gc.collect(); torch.cuda.empty_cache(); print("predicted:", r["trial_id"], "seed", s)
        except Exception as e:
            print(f"\nFAILED on {r['trial_id']} seed {s}: {type(e).__name__}: {str(e)[:160]}")
            if "illegal memory" in str(e) or "CUDA error" in str(e): print("CUDA context is broken: Kernel > Restart, rerun Steps 0 to 4 and 7 (finished predictions are skipped).")
            raise
print("predictions available:", sum(os.path.exists(lpath(r["trial_id"], s)) for r in TR.to_dict("records") for s in SEEDS))

In [ ]:
# STAGE 3 / STEP 8 - evaluation (CPU only): how reliable is source-only selection, and does any tuned DG method beat tuned ERM on the unseen site?
lpath = lambda tid, s: rf"{PRED}\logits_{tid}_seed{s}.npz"
Yd = {T: g["Y"] for T, g in TG.items()}; grp = {T: g["grp"] for T, g in TG.items()}
LGZ = {}
for r in TR.to_dict("records"):
    for s in SEEDS:
        if os.path.exists(lpath(r["trial_id"], s)): z = np.load(lpath(r["trial_id"], s)); LGZ[(r["trial_id"], s)] = {T: z[T] for T in TG}

# 8A: every trial (seed 0): selection signal versus the three test sites
rows = []
for r in TRS.to_dict("records"):
    z = LGZ.get((r["trial_id"], 0))
    if z is None: continue
    a5 = {T: np.mean([fauc(Yd[T][:, j], z[T][:, j]) for j in IDX5]) for T in TG}; rows.append(dict(r, auroc_A=a5["A"], auroc_B=a5["B"], auroc_C=a5["C"]))
TRR = pd.DataFrame(rows); core = TRR[~TRR.diagnostic]; TRR.to_csv(rf"{S3}\trials_all_results.csv", index=False)
rho_all = spearmanr(core.val_mean, core.auroc_C)[0]; rho_m = {m: spearmanr(core[core.method == m].val_mean, core[core.method == m].auroc_C)[0] for m in METHODS}
ORA = {m: core[core.method == m].sort_values("auroc_C").iloc[-1].to_dict() for m in METHODS}
sel0 = {m: TRR[TRR.trial_id == SEL[m]["trial_id"]].iloc[0].to_dict() for m in METHODS}
dft0 = {m: TRR[TRR.trial_id == f"{m.lower()}_t0"].iloc[0].to_dict() for m in METHODS}
REG = pd.DataFrame([dict(method=m, default_C=dft0[m]["auroc_C"], selected_C=sel0[m]["auroc_C"], oracle_C=ORA[m]["auroc_C"], regret_oracle_minus_selected=ORA[m]["auroc_C"] - sel0[m]["auroc_C"],
                         tuning_gain_selected_minus_default=sel0[m]["auroc_C"] - dft0[m]["auroc_C"], spread_C_over_trials=core[core.method == m].auroc_C.max() - core[core.method == m].auroc_C.min(),
                         spearman_val_vs_C=rho_m[m], oracle_trial=ORA[m]["trial_id"], selected_trial=SEL[m]["trial_id"]) for m in METHODS])
display(REG.round(4)); REG.to_csv(rf"{S3}\selection_reliability.csv", index=False)
print(f"Spearman correlation between the source-validation score and the VinBigData AUROC over all {len(core)} trials: {rho_all:.3f}")
print("note: all numbers in this table are single-seed (seed 0), so they are descriptive; the oracle column leaks the target and is an upper bound, not a result.")

# 8B: paired bootstrap of the tuned methods (3 seeds each). Same tie-safe engine as stage 2.
FL = {T: {f"{m} (tuned)": {s: LGZ[(SEL[m]["trial_id"], s)][T] for s in SEEDS} for m in METHODS} for T in TG}
class Engine:
    """AUROC of any family under bootstrap weights c (one weight per image). Tied scores get half credit, exactly like the rank-based AUROC."""
    def __init__(self, Y): self.Y, self.cache = Y, {}
    def _prep(self, T, name, s, j):
        k = (T, name, s, j)
        if k not in self.cache:
            sc = FL[T][name][s][:, j]; o = np.argsort(sc, kind="stable"); ss = sc[o]; gid = np.cumsum(np.r_[True, ss[1:] != ss[:-1]]) - 1
            self.cache[k] = (o, self.Y[o, j].astype(float), gid, int(gid[-1]) + 1)
        return self.cache[k]
    def auroc(self, T, name, c):
        au = np.zeros(5)
        for jj, j in enumerate(IDX5):
            a = 0.0
            for s in SEEDS:
                o, ys, gid, ng = self._prep(T, name, s, j); cs = c[o]; pos = cs * ys; neg = cs - pos; P, Nn = pos.sum(), neg.sum()
                gneg = np.bincount(gid, weights=neg, minlength=ng); gpos = np.bincount(gid, weights=pos, minlength=ng); a += (gpos * (np.cumsum(gneg) - gneg + 0.5 * gneg)).sum() / (P * Nn)
            au[jj] = a / len(SEEDS)
        return au
def counts(T, rng):
    g = grp[T]; N = len(Yd[T])
    if g is None: return np.bincount(rng.integers(0, N, N), minlength=N).astype(float)
    G = g.max() + 1; return np.bincount(rng.integers(0, G, G), minlength=G)[g].astype(float)
ENG = {T: Engine(Yd[T]) for T in TG}
for T in TG:
    n0 = "ERM (tuned)"; fast = ENG[T].auroc(T, n0, np.ones(len(Yd[T]))); slow = np.array([np.mean([fauc(Yd[T][:, j], FL[T][n0][s][:, j]) for s in SEEDS]) for j in IDX5])
    assert np.abs(fast - slow).max() < 1e-9, f"engine check failed on {T}"
print("engine check passed")
NAMES = [f"{m} (tuned)" for m in METHODS]; BOOT = {}
for T in TG:
    rng = np.random.default_rng(PLAN["seed"]); arr = {n: np.zeros((NBOOT, 5)) for n in NAMES}
    for b in tqdm(range(NBOOT), desc=f"bootstrap {TG[T]['name']}"):
        c = counts(T, rng)
        for n in NAMES: arr[n][b] = ENG[T].auroc(T, n, c)
    BOOT[T] = arr
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); mm = len(p); adj = np.empty(mm); run = 0.0
    for r_, i in enumerate(o): run = max(run, (mm - r_) * p[i]); adj[i] = min(run, 1.0)
    return adj
rows = []
for T in ["C", "A", "B"]:
    for m in ["DANN", "CORAL", "GroupDRO"]:
        d = np.nanmean(BOOT[T][f"{m} (tuned)"], 1) - np.nanmean(BOOT[T]["ERM (tuned)"], 1); d = d[np.isfinite(d)]; lo, hi = np.percentile(d, [2.5, 97.5])
        rows.append(dict(test_site=TG[T]["name"], target=T, method=m, diff_vs_tuned_ERM=float(d.mean()), lo=lo, hi=hi, se=float(d.std(ddof=1)), p_raw=(min((d <= 0).sum(), (d >= 0).sum()) * 2 + 1) / (len(d) + 1)))
CT = pd.DataFrame(rows); CT["p_holm"] = np.concatenate([holm(CT[CT.target == T].p_raw.values) for T in ["C", "A", "B"]])
CT["verdict"] = np.where((CT["diff_vs_tuned_ERM"] >= DELTA) & (CT.p_holm < ALPHA), "BETTER", np.where((CT["diff_vs_tuned_ERM"] < 0) & (CT.p_holm < ALPHA), "WORSE", "NOT DISTINGUISHABLE"))
display(CT.round(4)); CT.to_csv(rf"{S3}\tuned_vs_ERM_contrasts.csv", index=False)
primary = CT[CT.target == "C"].set_index("method"); MDE = 2.8 * float(primary.se.median())
OVERALL = "DG_HELPS" if (primary.verdict == "BETTER").any() else ("ERM_BETTER" if (primary.verdict == "WORSE").all() else "ERM_NOT_BEATEN")
print(f"\nPRIMARY RESULT (VinBigData, unseen): {OVERALL} | minimal detectable difference about {MDE:.4f} (the plan requires {DELTA})")
rows = []
for T in TG:
    for n in NAMES: lo, hi = np.percentile(np.nanmean(BOOT[T][n], 1), [2.5, 97.5]); rows.append(dict(test_site=TG[T]["name"], model=n, AUROC=np.mean([np.mean([fauc(Yd[T][:, j], FL[T][n][s][:, j]) for j in IDX5]) for s in SEEDS]), CI=f"[{lo:.3f}, {hi:.3f}]"))
TUN = pd.DataFrame(rows); display(TUN.round(4)); TUN.to_csv(rf"{S3}\tuned_models_auroc.csv", index=False)

# 8C: what the GroupDRO fix changes (default settings, seed 0)
if "groupdro_raw_default" in set(TRR.trial_id):
    a, b_ = TRR[TRR.trial_id == "groupdro_raw_default"].iloc[0], TRR[TRR.trial_id == "groupdro_t0"].iloc[0]
    DRO = pd.DataFrame({"original GroupDRO (raw loss)": [a.val_mean, a.auroc_A, a.auroc_B, a.auroc_C], "fixed GroupDRO (relative hardness)": [b_.val_mean, b_.auroc_A, b_.auroc_B, b_.auroc_C]}, index=["source validation", "NIH test", "CheXpert test", "VinBigData"])
    display(DRO.round(4)); DRO.to_csv(rf"{S3}\groupdro_fix_effect.csv")